# Deep Learning Lab — Assignment 8
## Neural Language Translation: Attention Networks, Self-Attention and Multi-Head Attention

In [ ]:
# ── Global seed (DO NOT MOVE OR CHANGE) ──────────────────────────────────────
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import re, math
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

---
## Task 1: Data Preparation

In [ ]:
# Task 1 — YOUR CODE HERE

---
## Task 2: Additive (Bahdanau) Attention Seq2Seq

The skeleton for `AdditiveAttention`, `DecoderAdditive`, and `Seq2SeqAdditive` is provided below.  
**Your job:** Fill in all `# YOUR CODE HERE` sections.

In [ ]:
# ── Hyperparameters (DO NOT CHANGE) ───────────────────────────────────────────
EMB_DIM   = 64
HID_DIM   = 128
N_EPOCHS  = 5
LR        = 1e-3
SRC_VOCAB = len(src_stoi)
TGT_VOCAB = len(tgt_stoi)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# ── Encoder (DO NOT CHANGE) ───────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        # src      : (B, 30)
        # embedded : (B, 30, 64)
        # outputs  : (B, 30, 128)
        # hidden   : (1, B, 128)
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

In [ ]:
# ── AdditiveAttention — 
class AdditiveAttention(nn.Module):
    """
    Bahdanau additive attention.
    Formula: e_{t,i} = v^T tanh( W_h*h_i + W_s*s_t + b )
    """
    def __init__(self, hid_dim):
        super().__init__()
        # Learnable parameters — DO NOT CHANGE
        self.Wh = nn.Linear(hid_dim, hid_dim, bias=False)  # projects encoder states
        self.Ws = nn.Linear(hid_dim, hid_dim, bias=True)   # projects decoder state + bias b
        self.v  = nn.Linear(hid_dim, 1,       bias=False)  # collapses to scalar score

    def forward(self, st, H, src_mask):
        """
        Inputs:
            st      : (B, 128)     — current decoder hidden state
            H       : (B, S, 128)  — all encoder hidden states
            src_mask: (B, S) bool  — True=valid token, False=<pad>
        Returns:
            context : (B, 128)     — weighted sum of encoder states
            alpha   : (B, S)       — attention weights (sum to 1 over S)
        """
        # Step 1: Compute energy scores
        # Hint: self.Wh(H) → (B,S,128)  and  self.Ws(st) → (B,128)
        # Hint: unsqueeze st before adding so shapes match
        # Hint: apply tanh after addition
        # energy shape: (B, S, 128)
        energy = # YOUR CODE HERE

        # Step 2: Collapse to scalar scores using self.v
        # Hint: self.v(energy) → (B, S, 1) → squeeze last dim → (B, S)
        scores = # YOUR CODE HERE

        # Step 3: Mask padded positions
        # Hint: scores.masked_fill(~src_mask, float('-inf'))
        scores = # YOUR CODE HERE

        # Step 4: Normalize with softmax over S dimension
        # alpha shape: (B, S)
        alpha = # YOUR CODE HERE

        # Step 5: Compute context as weighted sum of encoder states
        # Hint: torch.bmm(alpha.unsqueeze(1), H) → (B,1,128) → squeeze → (B,128)
        context = # YOUR CODE HERE

        return context, alpha

In [ ]:
# ── DecoderAdditive — YOUR CODE HERE ─────────────────────────────────────────
class DecoderAdditive(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        # Layers — DO NOT CHANGE
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = AdditiveAttention(hid_dim)
        self.gru       = nn.GRUCell(emb_dim + hid_dim, hid_dim)
        self.fc_out    = nn.Linear(hid_dim * 2 + emb_dim, vocab_size)

    def forward_step(self, token, hidden, enc_out, src_mask):
        """
        Inputs:
            token   : (B,)         — current input token IDs
            hidden  : (B, 128)     — current decoder hidden state
            enc_out : (B, S, 128)  — all encoder hidden states
            src_mask: (B, S) bool  — padding mask
        Returns:
            pred   : (B, vocab_size) — scores over target vocabulary
            hidden : (B, 128)        — updated hidden state
            alpha  : (B, S)          — attention weights
        """
        # Step 1: Convert token IDs to embeddings
        # Hint: use self.embedding
        # emb shape: (B, 64)
        emb = # YOUR CODE HERE

        # Step 2: Compute context vector using attention
        # Hint: use self.attention(hidden, enc_out, src_mask)
        # context shape: (B, 128)   alpha shape: (B, S)
        context, alpha = # YOUR CODE HERE

        # Step 3: Concatenate emb and context, pass through GRUCell
        # Hint: torch.cat([emb, context], dim=-1) → (B, 192)
        # Hint: self.gru(gru_in, hidden) → new hidden (B, 128)
        gru_in = # YOUR CODE HERE
        hidden = # YOUR CODE HERE

        # Step 4: Concatenate hidden, context, emb and predict next word
        # Hint: torch.cat([hidden, context, emb], dim=-1) → (B, 320)
        # Hint: self.fc_out(...) → (B, vocab_size)
        pred = # YOUR CODE HERE

        return pred, hidden, alpha

In [ ]:
# ── Seq2SeqAdditive — YOUR CODE HERE ─────────────────────────────────────────
class Seq2SeqAdditive(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, src_mask):
        """
        Inputs:
            src     : (B, 30)  — source token IDs
            tgt     : (B, 30)  — target token IDs
            src_mask: (B, 30)  — source padding mask
        Returns:
            outputs : (B, 30, tgt_vocab_size) — predictions at every timestep
        """
        B, T = tgt.shape

        # Step 1: Pass source through encoder
        # Hint: self.encoder(src) returns enc_out (B,30,128) and hidden (1,B,128)
        # Hint: squeeze hidden to (B, 128)
        enc_out, hidden = # YOUR CODE HERE
        hidden = # YOUR CODE HERE

        # Step 2: Pre-allocate output tensor
        # shape: (B, T, TGT_VOCAB)
        outputs = torch.zeros(B, T, TGT_VOCAB).to(self.device)

        # Step 3: First input token is <sos> — tgt[:, 0]
        token = tgt[:, 0]   # (B,)

        # Step 4: Decode step by step
        for t in range(1, T):
            # Call decoder one step at a time
            # Hint: self.decoder.forward_step(token, hidden, enc_out, src_mask)
            # Store prediction in outputs[:, t]
            # Next input token = model's own prediction (argmax)
            pred, hidden, _ = # YOUR CODE HERE
            outputs[:, t]   = # YOUR CODE HERE
            token           = # YOUR CODE HERE

        return outputs

In [ ]:
# ── Build and train Task-2 model (DO NOT CHANGE) ──────────────────────────────
random.seed(42); np.random.seed(42); torch.manual_seed(42)

enc2   = Encoder(SRC_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
dec2   = DecoderAdditive(TGT_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
model2 = Seq2SeqAdditive(enc2, dec2, DEVICE).to(DEVICE)



---
## Task 3: Visualising Word–Word Relationships

In [ ]:
# Task 3 — YOUR CODE HERE

---
## Task 4: Scaled Dot-Product Attention

In [ ]:
# Task 4 — YOUR CODE HERE

---
## Task 5: Multi-Head Attention

In [ ]:
# Task 5 — YOUR CODE HERE

---
## Task 6: Comparison and Analysis

In [ ]:
# Task 6 — YOUR CODE HERE